# Week 7 EXERCISE: Code Review Severity Classifier

## Overview

Build a **QLoRA fine-tuning pipeline** for classifying code review comments into severity levels.

**Use Case:** Help development teams prioritize code review feedback by automatically classifying comments as **Critical**, **Major**, **Minor**, or **Suggestion**.

## Key Learnings from Week 7

- **Day 1**: QLoRA fundamentals - quantization + LoRA adapters
- **Day 2**: Prompt data preparation and base model selection
- **Day 3-4**: Training with QLoRA on GPU (Google Colab)
- **Day 5**: Evaluation and model comparison

## Pipeline Steps

1. **Generate Dataset** - Synthetic code review comments with severity labels
2. **Baseline Evaluation** - Zero-shot frontier model performance
3. **Prepare Training Data** - Format for QLoRA fine-tuning
4. **Export to HuggingFace** - Push dataset for Colab training
5. **Gradio UI** - Interactive classifier with evaluation

---

## Severity Levels

| Level | Description | Examples |
|-------|-------------|----------|
| **Critical** | Security vulnerability, data loss risk, crash | SQL injection, unhandled null, memory leak |
| **Major** | Bug, broken functionality, performance issue | Wrong logic, missing validation, N+1 query |
| **Minor** | Code smell, maintainability concern | Magic numbers, long function, missing docs |
| **Suggestion** | Style preference, optional improvement | Naming convention, formatting, refactor idea |

In [ ]:
# Imports

import os
import json
import random
import re
from collections import Counter
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)

In [ ]:
# Configuration

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
FRONTIER_MODEL = "openai/gpt-4.1-mini"

# Try OpenRouter first, fall back to OpenAI
api_key = os.getenv("OPENROUTER_API_KEY")
if api_key:
    client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=api_key)
    print("Using OpenRouter API")
else:
    client = OpenAI()
    FRONTIER_MODEL = "gpt-4.1-mini"
    print("Using OpenAI API directly")

## Dataset: Code Review Comments

Synthetic code review feedback with severity classifications.

In [ ]:
# Severity levels
SEVERITIES = ("Critical", "Major", "Minor", "Suggestion")

# Synthetic code review dataset
REVIEWS = [
    # CRITICAL - Security, crashes, data loss
    {"comment": "This SQL query is vulnerable to injection. User input is concatenated directly into the query string without parameterization.", "severity": "Critical"},
    {"comment": "The password is being logged in plaintext here. This is a serious security violation that could expose user credentials.", "severity": "Critical"},
    {"comment": "No null check before dereferencing this pointer. This will cause a segfault when the object is None.", "severity": "Critical"},
    {"comment": "API key is hardcoded in the source code. This should be moved to environment variables immediately.", "severity": "Critical"},
    {"comment": "This DELETE endpoint has no authentication check. Anyone can delete any record without authorization.", "severity": "Critical"},
    {"comment": "Race condition in the transaction handler. Two concurrent requests could withdraw more than the available balance.", "severity": "Critical"},
    {"comment": "XSS vulnerability: user input is rendered directly in HTML without escaping. Attackers could inject malicious scripts.", "severity": "Critical"},
    {"comment": "The encryption key is derived from a predictable value. This makes the encryption trivially breakable.", "severity": "Critical"},
    {"comment": "Buffer overflow risk: memcpy with user-controlled size parameter and no bounds checking.", "severity": "Critical"},
    {"comment": "Session tokens are stored in localStorage which is vulnerable to XSS. Use httpOnly cookies instead.", "severity": "Critical"},
    {"comment": "This recursive function has no base case termination. It will cause stack overflow on any input.", "severity": "Critical"},
    {"comment": "Database connection is never closed in the finally block. This will exhaust the connection pool under load.", "severity": "Critical"},
    {"comment": "CSRF protection is disabled for this form. Attackers can forge requests on behalf of authenticated users.", "severity": "Critical"},
    {"comment": "Private key file has 777 permissions. This exposes cryptographic material to all users on the system.", "severity": "Critical"},
    {"comment": "Integer overflow possible when calculating buffer size. Could lead to heap corruption.", "severity": "Critical"},
    
    # MAJOR - Bugs, broken functionality, performance
    {"comment": "The comparison uses = instead of == which will always evaluate to true and assign rather than compare.", "severity": "Major"},
    {"comment": "This loop fetches each item individually causing N+1 query problem. Should use batch fetch or JOIN.", "severity": "Major"},
    {"comment": "Return value is ignored here. The function returns an error code that should be checked.", "severity": "Major"},
    {"comment": "Off-by-one error in the loop boundary. The last element will never be processed.", "severity": "Major"},
    {"comment": "The cache has no expiration policy. Stale data will be served indefinitely after the first request.", "severity": "Major"},
    {"comment": "Exception is caught but error is swallowed silently. Add logging or re-throw to avoid hiding failures.", "severity": "Major"},
    {"comment": "Missing input validation. Negative values will cause incorrect calculations in the price computation.", "severity": "Major"},
    {"comment": "This regex will cause catastrophic backtracking on certain inputs, leading to ReDoS vulnerability.", "severity": "Major"},
    {"comment": "The else branch returns undefined instead of an empty array, breaking the contract with callers.", "severity": "Major"},
    {"comment": "Async operation without await. The subsequent code assumes the operation completed but it hasn't.", "severity": "Major"},
    {"comment": "State mutation inside render method. This will cause infinite re-render loop in React.", "severity": "Major"},
    {"comment": "The timeout is set to 0ms which effectively disables the retry mechanism entirely.", "severity": "Major"},
    {"comment": "Default case is missing in the switch statement. Unexpected values will fall through silently.", "severity": "Major"},
    {"comment": "The file handle is opened but never closed if an exception occurs. Use context manager or try-finally.", "severity": "Major"},
    {"comment": "Date parsing assumes UTC but user input is in local timezone. This will cause off-by-hours bugs.", "severity": "Major"},
    {"comment": "Floating point comparison using == will fail for calculated values due to precision issues.", "severity": "Major"},
    
    # MINOR - Code smells, maintainability
    {"comment": "Magic number 86400 should be extracted to a named constant like SECONDS_PER_DAY for clarity.", "severity": "Minor"},
    {"comment": "This function is 200 lines long. Consider breaking it into smaller, focused helper functions.", "severity": "Minor"},
    {"comment": "The variable name 'x' is not descriptive. Rename to something meaningful like 'userCount' or 'index'.", "severity": "Minor"},
    {"comment": "Duplicate code block appears in 3 places. Extract to a shared utility function to reduce repetition.", "severity": "Minor"},
    {"comment": "This class has 15 dependencies injected. Consider splitting into smaller classes with single responsibility.", "severity": "Minor"},
    {"comment": "The boolean parameter makes call sites unclear. Consider using named arguments or separate methods.", "severity": "Minor"},
    {"comment": "Dead code: this else branch can never be reached based on the preceding conditions.", "severity": "Minor"},
    {"comment": "Missing docstring for this public API function. Add documentation explaining parameters and return value.", "severity": "Minor"},
    {"comment": "Cyclomatic complexity is 25. Refactor to reduce nesting and improve testability.", "severity": "Minor"},
    {"comment": "The TODO comment here is 2 years old. Either address it or remove if no longer relevant.", "severity": "Minor"},
    {"comment": "Inconsistent error message format. Other endpoints return {error: msg} but this returns {message: msg}.", "severity": "Minor"},
    {"comment": "This import is unused. Remove to keep the imports section clean and reduce bundle size.", "severity": "Minor"},
    {"comment": "Hard-coded string 'production' should use an enum or constant for environment names.", "severity": "Minor"},
    {"comment": "The test file is missing. Add unit tests for this new functionality before merging.", "severity": "Minor"},
    {"comment": "Type annotation is missing for the return value. Add -> Optional[User] for better IDE support.", "severity": "Minor"},
    {"comment": "Deeply nested callbacks here. Consider refactoring to async/await for better readability.", "severity": "Minor"},
    
    # SUGGESTION - Style, optional improvements
    {"comment": "Consider using f-strings instead of .format() for better readability in Python 3.6+.", "severity": "Suggestion"},
    {"comment": "This could be simplified to a list comprehension: [x.id for x in items if x.active]", "severity": "Suggestion"},
    {"comment": "The variable name follows camelCase but the codebase uses snake_case. Consider renaming for consistency.", "severity": "Suggestion"},
    {"comment": "Might be cleaner to use early return here instead of wrapping everything in an if block.", "severity": "Suggestion"},
    {"comment": "Optional: you could use dataclasses here to reduce boilerplate in the __init__ method.", "severity": "Suggestion"},
    {"comment": "The comments explain what the code does but not why. Consider adding context about the business logic.", "severity": "Suggestion"},
    {"comment": "This utility function might be useful in other modules. Consider moving to a shared helpers file.", "severity": "Suggestion"},
    {"comment": "Nit: extra blank line here doesn't match the style in the rest of the file.", "severity": "Suggestion"},
    {"comment": "The ternary expression is getting long. Might be more readable as a simple if-else block.", "severity": "Suggestion"},
    {"comment": "Consider adding a type alias for this complex generic type to improve readability.", "severity": "Suggestion"},
    {"comment": "You could use destructuring assignment here: const { name, email } = user;", "severity": "Suggestion"},
    {"comment": "Optional chaining (?.) could simplify this null check chain.", "severity": "Suggestion"},
    {"comment": "The function name 'process' is generic. Something like 'validateAndTransform' would be more descriptive.", "severity": "Suggestion"},
    {"comment": "For future extensibility, you might want to use a strategy pattern here instead of switch.", "severity": "Suggestion"},
    {"comment": "Trailing commas in multiline arrays/objects make diffs cleaner. Optional but nice to have.", "severity": "Suggestion"},
    {"comment": "This enum could benefit from explicit string values for better debugging: Status.ACTIVE = 'active'", "severity": "Suggestion"},
    {"comment": "Consider grouping related imports together: stdlib, third-party, then local modules.", "severity": "Suggestion"},
    {"comment": "A brief README in this folder would help newcomers understand the module structure.", "severity": "Suggestion"},
]

print(f"Total reviews in dataset: {len(REVIEWS)}")
print(f"Severity distribution: {Counter(r['severity'] for r in REVIEWS)}")

## Train/Validation/Test Split

In [ ]:
# Shuffle and split (70% train, 15% val, 15% test)
random.seed(42)
shuffled = REVIEWS.copy()
random.shuffle(shuffled)

n = len(shuffled)
n_train = int(0.7 * n)
n_val = int(0.15 * n)

train_data = shuffled[:n_train]
val_data = shuffled[n_train:n_train + n_val]
test_data = shuffled[n_train + n_val:]

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")
print(f"\nTrain distribution: {Counter(r['severity'] for r in train_data)}")
print(f"Test distribution: {Counter(r['severity'] for r in test_data)}")

## Zero-Shot Baseline Evaluation

Measure frontier model performance without fine-tuning.

In [ ]:
# System prompt for classification
SYSTEM_PROMPT = """You are a code review assistant that classifies the severity of code review comments.
Classify each comment into exactly one severity level:

- Critical: Security vulnerability, crash risk, data loss, authentication bypass
- Major: Bug, broken functionality, performance issue, missing error handling
- Minor: Code smell, maintainability concern, missing docs, dead code
- Suggestion: Style preference, optional improvement, naming convention

Respond with ONLY the severity level (Critical, Major, Minor, or Suggestion), nothing else."""

def predict_baseline(comment: str) -> str:
    """Zero-shot classification via OpenRouter/OpenAI."""
    try:
        response = client.chat.completions.create(
            model=FRONTIER_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Classify this code review comment:\n\n{comment}"}
            ],
            max_tokens=10,
            temperature=0
        )
        if response.choices:
            raw = (response.choices[0].message.content or "").strip()
            for level in SEVERITIES:
                if level.lower() in raw.lower():
                    return level
            return raw or "Minor"
        return "Minor"
    except Exception as e:
        print(f"Prediction error: {e}")
        return "Minor"

# Test single prediction
test_review = test_data[0]
prediction = predict_baseline(test_review["comment"])
print(f"Sample comment: {test_review['comment'][:80]}...")
print(f"True: {test_review['severity']} | Predicted: {prediction}")

In [ ]:
def calculate_accuracy(predictor, data, verbose=False):
    """Calculate accuracy on a dataset."""
    correct = 0
    results = []
    
    for item in data:
        pred = predictor(item["comment"])
        is_correct = pred == item["severity"]
        if is_correct:
            correct += 1
        results.append({
            "comment": item["comment"][:60] + "...",
            "true": item["severity"],
            "predicted": pred,
            "correct": is_correct
        })
        if verbose:
            status = "OK" if is_correct else "WRONG"
            print(f"[{status}] True: {item['severity']:10} | Pred: {pred:10} | {item['comment'][:50]}...")
    
    accuracy = correct / len(data) if data else 0.0
    return accuracy, results

# Evaluate baseline on test set
print("Evaluating baseline model on test set...\n")
baseline_accuracy, baseline_results = calculate_accuracy(predict_baseline, test_data, verbose=True)
print(f"\n=== Baseline Accuracy: {baseline_accuracy:.1%} ===")
print("This is the number to beat with QLoRA fine-tuning!")

## Prepare Training Data for QLoRA

Format data for fine-tuning with chat template.

In [ ]:
# Training prompt format
TRAIN_SYSTEM_MSG = """Classify the severity of code review comments.
Respond with exactly one word: Critical, Major, Minor, or Suggestion."""

def format_for_training(item):
    """Format item for QLoRA fine-tuning."""
    return {
        "text": f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{TRAIN_SYSTEM_MSG}<|eot_id|><|start_header_id|>user<|end_header_id|>

Classify this code review comment:
{item['comment']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{item['severity']}<|eot_id|>""",
        "comment": item["comment"],
        "severity": item["severity"]
    }

# Preview formatted training example
sample = format_for_training(train_data[0])
print("=== Formatted Training Example ===")
print(sample["text"])

In [ ]:
# Format all data
train_formatted = [format_for_training(item) for item in train_data]
val_formatted = [format_for_training(item) for item in val_data]
test_formatted = [format_for_training(item) for item in test_data]

print(f"Formatted: {len(train_formatted)} train, {len(val_formatted)} val, {len(test_formatted)} test")

## Export Dataset to HuggingFace Hub

Push to HuggingFace for use in Google Colab training.

In [ ]:
# Export to JSONL for local use
JSONL_DIR = "jsonl"
os.makedirs(JSONL_DIR, exist_ok=True)

def write_jsonl(data, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

write_jsonl(train_formatted, f"{JSONL_DIR}/train.jsonl")
write_jsonl(val_formatted, f"{JSONL_DIR}/validation.jsonl")
write_jsonl(test_formatted, f"{JSONL_DIR}/test.jsonl")

print(f"Exported to {JSONL_DIR}/")
print(f"  - train.jsonl: {len(train_formatted)} examples")
print(f"  - validation.jsonl: {len(val_formatted)} examples")
print(f"  - test.jsonl: {len(test_formatted)} examples")

In [ ]:
# Optional: Push to HuggingFace Hub
# Uncomment and run if you want to train on Colab

'''
from datasets import Dataset, DatasetDict
from huggingface_hub import login

# Login to HuggingFace
hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    login(hf_token, add_to_git_credential=True)
    
    # Create dataset
    dataset = DatasetDict({
        "train": Dataset.from_list(train_formatted),
        "validation": Dataset.from_list(val_formatted),
        "test": Dataset.from_list(test_formatted)
    })
    
    # Push to Hub (change username)
    HF_USERNAME = "your-username"  # Change this!
    DATASET_NAME = f"{HF_USERNAME}/code-review-severity"
    dataset.push_to_hub(DATASET_NAME, private=True)
    print(f"Dataset pushed to: https://huggingface.co/datasets/{DATASET_NAME}")
else:
    print("HF_TOKEN not found. Set it to push to HuggingFace Hub.")
'''

## Generate More Training Data (Optional)

Use LLM to expand the dataset.

In [ ]:
def generate_reviews(n_per_severity: int = 5):
    """Generate additional synthetic code review comments."""
    prompt = f"""Generate {n_per_severity} realistic code review comments for each severity level.

Severity definitions:
- Critical: Security vulnerability, crash risk, data loss, authentication bypass
- Major: Bug, broken functionality, performance issue, missing error handling  
- Minor: Code smell, maintainability concern, missing docs, dead code
- Suggestion: Style preference, optional improvement, naming convention

Make them diverse across different programming languages and scenarios.

Reply with ONLY a JSON array, no other text:
[{{"comment": "review comment text", "severity": "Critical"}}, ...]"""
    
    try:
        response = client.chat.completions.create(
            model=FRONTIER_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=3000,
            temperature=0.8
        )
        if response.choices:
            raw = (response.choices[0].message.content or "").strip()
            raw = re.sub(r"^```(?:json)?\s*", "", raw).strip()
            raw = re.sub(r"\s*```$", "", raw).strip()
            
            generated = json.loads(raw)
            valid = [
                g for g in generated 
                if isinstance(g, dict) and g.get("severity") in SEVERITIES and g.get("comment")
            ]
            return valid
    except Exception as e:
        print(f"Generation failed: {e}")
    return []

# Uncomment to generate more data:
# new_reviews = generate_reviews(5)
# print(f"Generated {len(new_reviews)} new reviews")
# REVIEWS.extend(new_reviews)

## Gradio UI: Interactive Code Review Classifier

In [ ]:
# Severity color mapping
SEVERITY_COLORS = {
    "Critical": "#dc3545",   # Red
    "Major": "#fd7e14",      # Orange
    "Minor": "#ffc107",      # Yellow
    "Suggestion": "#28a745"  # Green
}

SEVERITY_DESCRIPTIONS = {
    "Critical": "Must fix before merge - security/crash risk",
    "Major": "Should fix - bug or significant issue",
    "Minor": "Nice to fix - code quality improvement",
    "Suggestion": "Optional - style or preference"
}

def classify_review(comment: str) -> str:
    """Classify a code review comment."""
    if not comment.strip():
        return "Please enter a code review comment."
    
    severity = predict_baseline(comment)
    color = SEVERITY_COLORS.get(severity, "#6c757d")
    desc = SEVERITY_DESCRIPTIONS.get(severity, "")
    
    return f"""## Classification Result

**Severity:** <span style="color:{color}; font-weight:bold; font-size:1.3em;">{severity}</span>

**Action:** {desc}

---

### Severity Guide:
- **Critical**: Security vulnerabilities, crashes, data loss
- **Major**: Bugs, broken functionality, performance issues
- **Minor**: Code smells, maintainability, missing docs
- **Suggestion**: Style preferences, optional improvements"""

def run_evaluation() -> str:
    """Run evaluation on test set."""
    accuracy, results = calculate_accuracy(predict_baseline, test_data)
    
    output = f"""## Evaluation Results

**Model:** {FRONTIER_MODEL}  
**Test Set Size:** {len(test_data)}  
**Accuracy:** {accuracy:.1%}

### Sample Predictions:

| Status | True | Predicted | Comment |
|--------|------|-----------|--------|
"""
    
    for r in results[:10]:
        status = "OK" if r["correct"] else "WRONG"
        output += f"| {status} | {r['true']} | {r['predicted']} | {r['comment'][:40]}... |\n"
    
    # Per-severity accuracy
    by_severity = {}
    for r in results:
        key = r["true"]
        if key not in by_severity:
            by_severity[key] = {"correct": 0, "total": 0}
        by_severity[key]["total"] += 1
        if r["correct"]:
            by_severity[key]["correct"] += 1
    
    output += "\n### Accuracy by Severity:\n\n"
    for sev in SEVERITIES:
        if sev in by_severity:
            stats = by_severity[sev]
            pct = stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
            output += f"- **{sev}**: {stats['correct']}/{stats['total']} ({pct:.0f}%)\n"
    
    return output

def get_sample_review(severity: str) -> str:
    """Get a random sample review for the selected severity."""
    matching = [r for r in REVIEWS if r["severity"] == severity]
    if matching:
        return random.choice(matching)["comment"]
    return ""

def show_export_info() -> str:
    """Show export information."""
    return f"""## Training Data Export

| File | Examples |
|------|----------|
| {JSONL_DIR}/train.jsonl | {len(train_formatted)} |
| {JSONL_DIR}/validation.jsonl | {len(val_formatted)} |
| {JSONL_DIR}/test.jsonl | {len(test_formatted)} |

### QLoRA Fine-Tuning Steps:

1. **Upload to HuggingFace Hub** (or use JSONL directly)
2. **Open Google Colab** with T4 GPU runtime
3. **Load base model** (e.g., Llama 3.2 3B) with 4-bit quantization
4. **Apply LoRA adapters** to attention layers
5. **Train** for 2-3 epochs with learning rate ~2e-4
6. **Evaluate** and compare to baseline

### Example Colab Setup:

```python
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

# 4-bit quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05
)
```
"""

In [ ]:
# Build Gradio Interface
with gr.Blocks(title="Code Review Classifier", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""# Code Review Severity Classifier
    
Classify code review comments into severity levels using LLM.
This demonstrates the baseline before QLoRA fine-tuning.""")
    
    with gr.Tabs():
        # Tab 1: Classify Review
        with gr.Tab("Classify Review"):
            with gr.Row():
                with gr.Column():
                    review_input = gr.Textbox(
                        label="Code Review Comment",
                        placeholder="Enter a code review comment to classify...",
                        lines=4
                    )
                    with gr.Row():
                        classify_btn = gr.Button("Classify", variant="primary")
                        clear_btn = gr.Button("Clear")
                    
                    gr.Markdown("### Load Sample Review:")
                    with gr.Row():
                        for sev in SEVERITIES:
                            btn = gr.Button(sev, size="sm")
                            btn.click(
                                fn=lambda s=sev: get_sample_review(s),
                                outputs=review_input
                            )
                
                with gr.Column():
                    result_output = gr.Markdown(label="Result")
            
            classify_btn.click(fn=classify_review, inputs=review_input, outputs=result_output)
            clear_btn.click(fn=lambda: ("", ""), outputs=[review_input, result_output])
        
        # Tab 2: Evaluation
        with gr.Tab("Evaluate Model"):
            gr.Markdown("""### Baseline Evaluation
            
Run the zero-shot classifier on the test set to measure baseline accuracy.
This is the number to beat with QLoRA fine-tuning!""")
            
            eval_btn = gr.Button("Run Evaluation", variant="primary")
            eval_output = gr.Markdown()
            
            eval_btn.click(fn=run_evaluation, outputs=eval_output)
        
        # Tab 3: Export Data
        with gr.Tab("QLoRA Training"):
            gr.Markdown("""### Fine-Tuning Instructions
            
Export training data and follow QLoRA fine-tuning steps.""")
            
            export_btn = gr.Button("Show Export Info", variant="primary")
            export_output = gr.Markdown()
            
            export_btn.click(fn=show_export_info, outputs=export_output)
        
        # Tab 4: Dataset Info
        with gr.Tab("Dataset Info"):
            dataset_info = f"""### Dataset Statistics

| Split | Count |
|-------|-------|
| Total | {len(REVIEWS)} |
| Train | {len(train_data)} |
| Validation | {len(val_data)} |
| Test | {len(test_data)} |

### Severity Distribution:

| Severity | Count | Percentage |
|----------|-------|------------|
"""
            counts = Counter(r['severity'] for r in REVIEWS)
            for sev in SEVERITIES:
                count = counts.get(sev, 0)
                pct = count / len(REVIEWS) * 100
                dataset_info += f"| {sev} | {count} | {pct:.1f}% |\n"
            
            dataset_info += f"""\n### Model Configuration

- **Baseline Model:** {FRONTIER_MODEL}
- **Recommended Base Model:** Llama 3.2 3B or Qwen 2.5 3B
- **QLoRA Rank:** 16
- **Target Modules:** q_proj, v_proj, k_proj, o_proj
"""
            gr.Markdown(dataset_info)

# Launch the interface
demo.launch()

## QLoRA Training Code (For Google Colab)

Copy this to a Colab notebook with T4 GPU.

In [ ]:
# === COLAB TRAINING CODE ===
# Copy everything below to a Google Colab notebook with T4 GPU

COLAB_CODE = '''
# Install dependencies
!pip install -q transformers peft bitsandbytes accelerate datasets huggingface_hub trl

import os
import torch
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

# Login to HuggingFace
login(userdata.get("HF_TOKEN"), add_to_git_credential=True)

# Configuration
BASE_MODEL = "meta-llama/Llama-3.2-3B"  # or "Qwen/Qwen2.5-3B"
HF_USERNAME = "your-username"  # Change this!
DATASET_NAME = f"{HF_USERNAME}/code-review-severity"
OUTPUT_MODEL = f"{HF_USERNAME}/code-review-classifier"

# Load dataset
dataset = load_dataset(DATASET_NAME)
print(f"Train: {len(dataset['train'])} | Val: {len(dataset['validation'])}")

# 4-bit quantization config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    fp16=True,
    report_to="none",
)

# Trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=512,
)

# Train!
trainer.train()

# Save to HuggingFace Hub
model.push_to_hub(OUTPUT_MODEL, private=True)
tokenizer.push_to_hub(OUTPUT_MODEL, private=True)
print(f"Model saved to: https://huggingface.co/{OUTPUT_MODEL}")
'''

print("=== Copy this code to Google Colab with T4 GPU ===")
print(COLAB_CODE)

## Summary

This exercise demonstrated:

1. **Dataset Creation**: Synthetic code review comments with severity labels
2. **Baseline Evaluation**: Zero-shot frontier model performance
3. **Training Data Preparation**: Formatted for QLoRA fine-tuning
4. **Export**: JSONL and HuggingFace Hub for Colab training
5. **Gradio UI**: Interactive classifier interface

### Expected Results

| Model | Expected Accuracy |
|-------|------------------|
| Random baseline | ~25% |
| GPT-4.1-mini (zero-shot) | ~60-75% |
| QLoRA fine-tuned Llama 3B | ~80-90% |

### Next Steps

1. Push dataset to HuggingFace Hub
2. Run QLoRA training on Google Colab (T4 GPU)
3. Compare fine-tuned model accuracy to baseline
4. Deploy fine-tuned model for offline inference